# 02 — Tensor Manipulation, Arithmetic, and Broadcasting (Sections 3, 4, 5)

This notebook covers:
- **Section 3:** Indexing/slicing, selecting rows/columns, reshaping, viewing dimensions, transposing, squeezing/unsqueezing, concatenating, stacking (100-value matrix exercises 1-11).
- **Section 4:** Element-wise ops, matrix multiplication, dot products, matrix-vector multiplication, sum/mean, min/max, argmax/argmin, reductions across dimensions (coding exercises 1-9).
- **Section 5:** Broadcasting rules/alignment/singleton dims, compatible/incompatible shapes, arithmetic/batch broadcasting (experiments 1-9 with predicted shapes and caught invalid cases).


In [1]:
import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)


## Section 3 — Indexing, Slicing, and Shape Manipulation
### Matrix with 100 numerical values

In [2]:
M = torch.arange(1, 101).reshape(10, 10)
print('M shape:', tuple(M.shape), 'numel:', M.numel())
print(M)


M shape: (10, 10) numel: 100
tensor([[  1,   2,   3,   4,   5,   6,   7,   8,   9,  10],
        [ 11,  12,  13,  14,  15,  16,  17,  18,  19,  20],
        [ 21,  22,  23,  24,  25,  26,  27,  28,  29,  30],
        [ 31,  32,  33,  34,  35,  36,  37,  38,  39,  40],
        [ 41,  42,  43,  44,  45,  46,  47,  48,  49,  50],
        [ 51,  52,  53,  54,  55,  56,  57,  58,  59,  60],
        [ 61,  62,  63,  64,  65,  66,  67,  68,  69,  70],
        [ 71,  72,  73,  74,  75,  76,  77,  78,  79,  80],
        [ 81,  82,  83,  84,  85,  86,  87,  88,  89,  90],
        [ 91,  92,  93,  94,  95,  96,  97,  98,  99, 100]])


#### 1. Extract its first row

In [3]:
first_row = M[0]
print('First row:', first_row)
print('Shape:', tuple(first_row.shape))


First row: tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10])
Shape: (10,)


#### 2. Extract its last column

In [4]:
last_col = M[:, -1]
print('Last column:', last_col)
print('Shape:', tuple(last_col.shape))


Last column: tensor([ 10,  20,  30,  40,  50,  60,  70,  80,  90, 100])
Shape: (10,)


#### 3. Extract a submatrix from selected rows and columns

In [5]:
submatrix = M[2:5, 1:4]
print('M[2:5, 1:4] (rows 2-4, cols 1-3):\n', submatrix)
print('Shape:', tuple(submatrix.shape))
rows = torch.tensor([0, 3, 9])
cols = torch.tensor([0, 4, 9])
selected = M[rows[:, None], cols]
print('Selected rows [0,3,9], cols [0,4,9]:\n', selected)
print('Shape:', tuple(selected.shape))


M[2:5, 1:4] (rows 2-4, cols 1-3):
 tensor([[22, 23, 24],
        [32, 33, 34],
        [42, 43, 44]])
Shape: (3, 3)
Selected rows [0,3,9], cols [0,4,9]:
 tensor([[  1,   5,  10],
        [ 31,  35,  40],
        [ 91,  95, 100]])
Shape: (3, 3)


#### 4. Reshape it into different valid shapes

In [6]:
shapes = [(20, 5), (5, 20), (4, 25), (2, 5, 10), (100,)]
for s in shapes:
  r = M.reshape(s)
  print('reshape', s, '->', tuple(r.shape), 'numel:', r.numel())
r6 = M.reshape(10, -1)
print('reshape (10, -1) ->', tuple(r6.shape))


reshape (20, 5) -> (20, 5) numel: 100
reshape (5, 20) -> (5, 20) numel: 100
reshape (4, 25) -> (4, 25) numel: 100
reshape (2, 5, 10) -> (2, 5, 10) numel: 100
reshape (100,) -> (100,) numel: 100
reshape (10, -1) -> (10, 10)


#### 5. Convert a vector into a row matrix
#### 6. Convert a vector into a column matrix

In [7]:
v = torch.tensor([1, 2, 3, 4])
row_matrix = v.unsqueeze(0)
col_matrix = v.unsqueeze(1)
print('v:', v, tuple(v.shape))
print('row matrix:\n', row_matrix, tuple(row_matrix.shape))
print('column matrix:\n', col_matrix, tuple(col_matrix.shape))
print('reshape(1,-1):', tuple(v.reshape(1, -1).shape), 'reshape(-1,1):', tuple(v.reshape(-1, 1).shape))


v: tensor([1, 2, 3, 4]) (4,)
row matrix:
 tensor([[1, 2, 3, 4]]) (1, 4)
column matrix:
 tensor([[1],
        [2],
        [3],
        [4]]) (4, 1)
reshape(1,-1): (1, 4) reshape(-1,1): (4, 1)


#### 7. Remove singleton dimensions (explicit singleton-dim setup)

In [8]:
x_singleton = torch.arange(12).reshape(1, 3, 1, 4)
print('Original with singleton dims shape:', tuple(x_singleton.shape))
print('squeeze() ->', tuple(x_singleton.squeeze().shape))
print('squeeze(0) ->', tuple(x_singleton.squeeze(0).shape))
print('squeeze(2) ->', tuple(x_singleton.squeeze(2).shape))


Original with singleton dims shape: (1, 3, 1, 4)
squeeze() -> (3, 4)
squeeze(0) -> (3, 1, 4)
squeeze(2) -> (1, 3, 4)


#### 8. Transpose the matrix

In [9]:
print('M shape:', tuple(M.shape))
print('M.T shape:', tuple(M.T.shape))
print('transpose(0,1) shape:', tuple(M.transpose(0, 1).shape))
print('Top-left 3x3 M:\n', M[:3, :3])
print('Top-left 3x3 M.T:\n', M.T[:3, :3])
print('Equal:', torch.equal(M.T, M.transpose(0, 1)))


M shape: (10, 10)
M.T shape: (10, 10)
transpose(0,1) shape: (10, 10)
Top-left 3x3 M:
 tensor([[ 1,  2,  3],
        [11, 12, 13],
        [21, 22, 23]])
Top-left 3x3 M.T:
 tensor([[ 1, 11, 21],
        [ 2, 12, 22],
        [ 3, 13, 23]])
Equal: True


#### 9. Concatenate two matrices along rows and columns

In [10]:
A = torch.arange(1, 7).reshape(2, 3)
B = torch.arange(7, 13).reshape(2, 3)
print('A:\n', A)
print('B:\n', B)
cat_rows = torch.cat([A, B], dim=0)
cat_cols = torch.cat([A, B], dim=1)
print('cat rows dim=0:\n', cat_rows, tuple(cat_rows.shape))
print('cat cols dim=1:\n', cat_cols, tuple(cat_cols.shape))


A:
 tensor([[1, 2, 3],
        [4, 5, 6]])
B:
 tensor([[ 7,  8,  9],
        [10, 11, 12]])
cat rows dim=0:
 tensor([[ 1,  2,  3],
        [ 4,  5,  6],
        [ 7,  8,  9],
        [10, 11, 12]]) (4, 3)
cat cols dim=1:
 tensor([[ 1,  2,  3,  7,  8,  9],
        [ 4,  5,  6, 10, 11, 12]]) (2, 6)


#### 10. Stack three vectors into a single tensor

In [11]:
v1 = torch.tensor([1, 2, 3])
v2 = torch.tensor([4, 5, 6])
v3 = torch.tensor([7, 8, 9])
stacked0 = torch.stack([v1, v2, v3], dim=0)
stacked1 = torch.stack([v1, v2, v3], dim=1)
print('v1,v2,v3:', v1, v2, v3)
print('stack dim=0:\n', stacked0, tuple(stacked0.shape))
print('stack dim=1:\n', stacked1, tuple(stacked1.shape))
print('cat 1D vectors dim=0 shape:', tuple(torch.cat([v1, v2, v3], dim=0).shape))


v1,v2,v3: tensor([1, 2, 3]) tensor([4, 5, 6]) tensor([7, 8, 9])
stack dim=0:
 tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]]) (3, 3)
stack dim=1:
 tensor([[1, 4, 7],
        [2, 5, 8],
        [3, 6, 9]]) (3, 3)
cat 1D vectors dim=0 shape: (9,)


#### 11. Explain why some reshaping operations are possible and others are not

Reshape is valid only when total element count is preserved (`numel` unchanged). Invalid cases are demonstrated with caught exceptions so execution continues.


In [12]:
print('M numel:', M.numel())
invalid_cases = [
  {'shape': (3, 30), 'reason': '3*30=90 != 100'},
  {'shape': (10, 11), 'reason': '10*11=110 != 100'},
]
for i, case in enumerate(invalid_cases, 1):
  print('\nInvalid reshape {}: {}'.format(i, case['shape']))
  print('Predicted: FAIL -', case['reason'])
  try:
    _ = M.reshape(case['shape'])
    print('Unexpected success')
  except Exception as e:
    print('Actual exception type:', type(e).__name__)
    print('Actual exception message:', str(e))


M numel: 100

Invalid reshape 1: (3, 30)
Predicted: FAIL - 3*30=90 != 100
Actual exception type: RuntimeError
Actual exception message: shape '[3, 30]' is invalid for input of size 100

Invalid reshape 2: (10, 11)
Predicted: FAIL - 10*11=110 != 100
Actual exception type: RuntimeError
Actual exception message: shape '[10, 11]' is invalid for input of size 100


## Section 4 — Tensor Arithmetic and Reductions
Study topics supported: element-wise add/sub/mul/div, matrix multiplication, dot products, matrix-vector multiplication, sum/mean, min/max, argmax/argmin, reduction across dimensions.


### 1. Perform element-wise arithmetic on two matrices

In [13]:
X = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
Y = torch.tensor([[6., 5., 4.], [3., 2., 1.]])
print('X:\n', X)
print('Y:\n', Y)
print('X+Y:\n', X + Y)
print('X-Y:\n', X - Y)
print('X*Y:\n', X * Y)
print('X/Y:\n', X / Y)
print('X**2:\n', X ** 2)


X:
 tensor([[1., 2., 3.],
        [4., 5., 6.]])
Y:
 tensor([[6., 5., 4.],
        [3., 2., 1.]])
X+Y:
 tensor([[7., 7., 7.],
        [7., 7., 7.]])
X-Y:
 tensor([[-5., -3., -1.],
        [ 1.,  3.,  5.]])
X*Y:
 tensor([[ 6., 10., 12.],
        [12., 10.,  6.]])
X/Y:
 tensor([[0.1667, 0.4000, 0.7500],
        [1.3333, 2.5000, 6.0000]])
X**2:
 tensor([[ 1.,  4.,  9.],
        [16., 25., 36.]])


### 2. Perform matrix multiplication on compatible matrices

In [14]:
A2 = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
B2 = torch.tensor([[7., 8.], [9., 10.], [11., 12.]])
mm = A2 @ B2
print('A2', tuple(A2.shape), 'B2', tuple(B2.shape))
print('A2@B2:\n', mm, tuple(mm.shape))
print('matmul equal:', torch.equal(mm, torch.matmul(A2, B2)))


A2 (2, 3) B2 (3, 2)
A2@B2:
 tensor([[ 58.,  64.],
        [139., 154.]]) (2, 2)
matmul equal: True


### 3. Calculate the dot product of two vectors

In [15]:
u = torch.tensor([1., 2., 3.])
v = torch.tensor([4., 5., 6.])
dot = torch.dot(u, v)
manual = (u * v).sum()
print('u:', u, 'v:', v)
print('dot:', dot, 'manual:', manual, 'close:', torch.allclose(dot, manual))


u: tensor([1., 2., 3.]) v: tensor([4., 5., 6.])
dot: tensor(32.) manual: tensor(32.) close: True


### Matrix-vector multiplication (study topic demo)

In [16]:
A_mv = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
x_mv = torch.tensor([1., 0., -1.])
y_mv = A_mv @ x_mv
print('A shape:', tuple(A_mv.shape), 'x shape:', tuple(x_mv.shape))
print('A @ x =', y_mv, 'shape:', tuple(y_mv.shape))


A shape: (2, 3) x shape: (3,)
A @ x = tensor([-2., -2.]) shape: (2,)


### 4. Calculate row-wise and column-wise sums
### 5. Calculate the mean of an entire tensor
### 6. Find maximum and minimum values along different dimensions
### 7. Identify the index of the maximum value in every row

In [17]:
T = torch.tensor([[1., 5., 2.], [4., 4., 9.], [7., 0., 3.]])
print('T:\n', T)
print('Row-wise sums dim=1:', T.sum(dim=1))
print('Column-wise sums dim=0:', T.sum(dim=0))
print('Mean of entire tensor:', T.mean().item())
max0_v, max0_i = T.max(dim=0)
min0_v, min0_i = T.min(dim=0)
max1_v, max1_i = T.max(dim=1)
min1_v, min1_i = T.min(dim=1)
print('Max dim=0 values:', max0_v, 'indices:', max0_i)
print('Min dim=0 values:', min0_v, 'indices:', min0_i)
print('Max dim=1 values:', max1_v, 'indices:', max1_i)
print('Min dim=1 values:', min1_v, 'indices:', min1_i)
print('Global max/min:', T.max().item(), T.min().item())
row_argmax = torch.argmax(T, dim=1)
print('Index of maximum value in every row:', row_argmax)


T:
 tensor([[1., 5., 2.],
        [4., 4., 9.],
        [7., 0., 3.]])
Row-wise sums dim=1: tensor([ 8., 17., 10.])
Column-wise sums dim=0: tensor([12.,  9., 14.])
Mean of entire tensor: 3.8888888359069824
Max dim=0 values: tensor([7., 5., 9.]) indices: tensor([2, 0, 1])
Min dim=0 values: tensor([1., 0., 2.]) indices: tensor([0, 2, 0])
Max dim=1 values: tensor([5., 9., 7.]) indices: tensor([1, 2, 0])
Min dim=1 values: tensor([1., 4., 0.]) indices: tensor([0, 0, 1])
Global max/min: 9.0 0.0
Index of maximum value in every row: tensor([1, 2, 0])


### 8. Reproduce a small numerical matrix calculation using both NumPy and PyTorch
### 9. Verify that the results match (explicit allclose tolerances)

In [18]:
A_np = np.array([[1., 2.], [3., 4.], [5., 6.]], dtype=np.float32)
B_np = np.array([[0.5, 1.0, -1.0], [2.0, 0.0, 1.0]], dtype=np.float32)
C_np = np.array([[1.], [0.], [-1.]], dtype=np.float32)
scale_np = np.float32(2.0)
Z_np = (A_np @ B_np + C_np) * scale_np
row_sums_np = Z_np.sum(axis=1)
col_sums_np = Z_np.sum(axis=0)
mean_np = Z_np.mean()
print('NumPy Z:\n', Z_np)
print('NumPy row sums:', row_sums_np, 'col sums:', col_sums_np, 'mean:', mean_np)

A_t = torch.from_numpy(A_np)
B_t = torch.from_numpy(B_np)
C_t = torch.from_numpy(C_np)
Z_t = (A_t @ B_t + C_t) * 2.0
row_sums_t = Z_t.sum(dim=1)
col_sums_t = Z_t.sum(dim=0)
mean_t = Z_t.mean()
print('PyTorch Z:\n', Z_t)
print('PyTorch row sums:', row_sums_t, 'col sums:', col_sums_t, 'mean:', mean_t)

atol, rtol = 1e-7, 1e-7
checks = {
  'Z np.allclose': np.allclose(Z_np, Z_t.detach().cpu().numpy(), atol=atol, rtol=rtol),
  'row sums np.allclose': np.allclose(row_sums_np, row_sums_t.detach().cpu().numpy(), atol=atol, rtol=rtol),
  'col sums np.allclose': np.allclose(col_sums_np, col_sums_t.detach().cpu().numpy(), atol=atol, rtol=rtol),
  'mean np.allclose': np.allclose(np.array(mean_np), np.array(mean_t.item()), atol=atol, rtol=rtol),
  'Z torch.allclose': torch.allclose(Z_t, torch.from_numpy(Z_np), atol=atol, rtol=rtol),
}
print('Tolerances atol={} rtol={}'.format(atol, rtol))
for k, v in checks.items():
  print(k, ':', v)
print('All match:', all(checks.values()))


NumPy Z:
 [[11.  4.  4.]
 [19.  6.  2.]
 [27.  8.  0.]]
NumPy row sums: [19. 27. 35.] col sums: [57. 18.  6.] mean: 9.0
PyTorch Z:
 tensor([[11.,  4.,  4.],
        [19.,  6.,  2.],
        [27.,  8.,  0.]])
PyTorch row sums: tensor([19., 27., 35.]) col sums: tensor([57., 18.,  6.]) mean: tensor(9.)
Tolerances atol=1e-07 rtol=1e-07
Z np.allclose : True
row sums np.allclose : True
col sums np.allclose : True
mean np.allclose : True
Z torch.allclose : True
All match: True


## Section 5 — Broadcasting
Study topics: broadcasting rules, dimension alignment, singleton dimensions, compatible/incompatible shapes, broadcasting in arithmetic and batch computations.


### 1. Add a scalar to a matrix
- Matrix `(3,4)`, scalar `()` -> **predicted (3,4)**

In [19]:
mat = torch.arange(12).reshape(3, 4)
scalar = torch.tensor(10)
print('mat', tuple(mat.shape), 'scalar', tuple(scalar.shape))
print('Predicted: (3, 4)')
out = mat + scalar
print('Actual:', tuple(out.shape))
print(out)


mat (3, 4) scalar ()
Predicted: (3, 4)
Actual: (3, 4)
tensor([[10, 11, 12, 13],
        [14, 15, 16, 17],
        [18, 19, 20, 21]])


### 2. Add a vector to every row of a matrix
- Matrix `(3,4)`, row vector `(4,)` -> **predicted (3,4)**

In [20]:
mat = torch.arange(12, dtype=torch.float32).reshape(3, 4)
row_vec = torch.tensor([1., 2., 3., 4.])
print('mat', tuple(mat.shape), 'row_vec', tuple(row_vec.shape))
print('Predicted: (3, 4)')
out = mat + row_vec
print('Actual:', tuple(out.shape))
print(out)


mat (3, 4) row_vec (4,)
Predicted: (3, 4)
Actual: (3, 4)
tensor([[ 1.,  3.,  5.,  7.],
        [ 5.,  7.,  9., 11.],
        [ 9., 11., 13., 15.]])


### 3. Add a column vector to every column of a matrix
- Matrix `(3,4)`, column vector `(3,1)` -> **predicted (3,4)**

In [21]:
mat = torch.arange(12, dtype=torch.float32).reshape(3, 4)
col_vec = torch.tensor([[10.], [20.], [30.]])
print('mat', tuple(mat.shape), 'col_vec', tuple(col_vec.shape))
print('Predicted: (3, 4)')
out = mat + col_vec
print('Actual:', tuple(out.shape))
print(out)


mat (3, 4) col_vec (3, 1)
Predicted: (3, 4)
Actual: (3, 4)
tensor([[10., 11., 12., 13.],
        [24., 25., 26., 27.],
        [38., 39., 40., 41.]])


### 4. Perform element-wise multiplication between compatible shapes
- `(2,3) * (1,3)` -> **predicted (2,3)**

In [22]:
a = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
b = torch.tensor([[10., 100., 1000.]])
print('a', tuple(a.shape), 'b', tuple(b.shape))
print('Predicted: (2, 3)')
out = a * b
print('Actual:', tuple(out.shape))
print(out)


a (2, 3) b (1, 3)
Predicted: (2, 3)
Actual: (2, 3)
tensor([[  10.,  200., 3000.],
        [  40.,  500., 6000.]])


### 5. Experiment with broadcasting across three dimensions
- `(2,3,4) + (1,4)` -> **predicted (2,3,4)**

In [23]:
x = torch.ones(2, 3, 4)
y = torch.arange(4, dtype=torch.float32).reshape(1, 4)
print('x', tuple(x.shape), 'y', tuple(y.shape))
print('Predicted: (2, 3, 4)')
out = x + y
print('Actual:', tuple(out.shape))
print(out[0])


x (2, 3, 4) y (1, 4)
Predicted: (2, 3, 4)
Actual: (2, 3, 4)
tensor([[1., 2., 3., 4.],
        [1., 2., 3., 4.],
        [1., 2., 3., 4.]])


### 6. Five examples of valid broadcasting (predict before execution)

In [24]:
valid_examples = [
  {'name': 'Valid 1: scalar + matrix', 'a': torch.tensor(5), 'b': torch.ones(2, 3), 'pred': (2, 3), 'op': lambda a,b: a+b},
  {'name': 'Valid 2: row vector + matrix', 'a': torch.ones(3, 4), 'b': torch.arange(4), 'pred': (3, 4), 'op': lambda a,b: a+b},
  {'name': 'Valid 3: column vector + matrix', 'a': torch.ones(3, 4), 'b': torch.arange(3).reshape(3,1), 'pred': (3, 4), 'op': lambda a,b: a+b},
  {'name': 'Valid 4: 3D + 2D', 'a': torch.ones(2, 3, 4), 'b': torch.ones(3, 1), 'pred': (2, 3, 4), 'op': lambda a,b: a+b},
  {'name': 'Valid 5: singleton 3D + singleton 3D', 'a': torch.ones(4, 1, 5), 'b': torch.ones(1, 3, 5), 'pred': (4, 3, 5), 'op': lambda a,b: a+b},
]
for ex in valid_examples:
  out = ex['op'](ex['a'], ex['b'])
  print(ex['name'])
  print('  a', tuple(ex['a'].shape), 'b', tuple(ex['b'].shape), 'pred', ex['pred'], 'actual', tuple(out.shape), 'match', tuple(out.shape)==ex['pred'])


Valid 1: scalar + matrix
  a () b (2, 3) pred (2, 3) actual (2, 3) match True
Valid 2: row vector + matrix
  a (3, 4) b (4,) pred (3, 4) actual (3, 4) match True
Valid 3: column vector + matrix
  a (3, 4) b (3, 1) pred (3, 4) actual (3, 4) match True
Valid 4: 3D + 2D
  a (2, 3, 4) b (3, 1) pred (2, 3, 4) actual (2, 3, 4) match True
Valid 5: singleton 3D + singleton 3D
  a (4, 1, 5) b (1, 3, 5) pred (4, 3, 5) actual (4, 3, 5) match True


### 7. Five examples of invalid broadcasting (predict before execution, caught failures)
### 8. Predicted output shapes before execution (included above/below)
### 9. Explain why the invalid examples fail

In [25]:
invalid_examples = [
  {'name': 'Invalid 1', 'a': torch.ones(3, 4), 'b': torch.ones(3), 'pred': 'FAIL', 'reason': 'Trailing dims 4 vs 3 (neither 1).'},
  {'name': 'Invalid 2', 'a': torch.ones(2, 3, 4), 'b': torch.ones(2, 2, 4), 'pred': 'FAIL', 'reason': 'Middle dims 3 vs 2 incompatible.'},
  {'name': 'Invalid 3', 'a': torch.ones(5, 4), 'b': torch.ones(4, 1), 'pred': 'FAIL', 'reason': 'Aligned dims 5 vs 4 conflict.'},
  {'name': 'Invalid 4', 'a': torch.ones(2, 2), 'b': torch.ones(2, 3), 'pred': 'FAIL', 'reason': 'Last dims 2 vs 3 incompatible.'},
  {'name': 'Invalid 5', 'a': torch.ones(4, 1, 3), 'b': torch.ones(1, 2, 2), 'pred': 'FAIL', 'reason': 'Last dims 3 vs 2 incompatible.'},
]
for ex in invalid_examples:
  print(ex['name'])
  print('  a', tuple(ex['a'].shape), 'b', tuple(ex['b'].shape))
  print('  predicted:', ex['pred'], '-', ex['reason'])
  try:
    out = ex['a'] + ex['b']
    print('  unexpected success', tuple(out.shape))
  except Exception as e:
    print('  actual exception type:', type(e).__name__)
    print('  actual exception message:', str(e))
  print()


Invalid 1
  a (3, 4) b (3,)
  predicted: FAIL - Trailing dims 4 vs 3 (neither 1).
  actual exception type: RuntimeError
  actual exception message: The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 1

Invalid 2
  a (2, 3, 4) b (2, 2, 4)
  predicted: FAIL - Middle dims 3 vs 2 incompatible.
  actual exception type: RuntimeError
  actual exception message: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1

Invalid 3
  a (5, 4) b (4, 1)
  predicted: FAIL - Aligned dims 5 vs 4 conflict.
  actual exception type: RuntimeError
  actual exception message: The size of tensor a (5) must match the size of tensor b (4) at non-singleton dimension 0

Invalid 4
  a (2, 2) b (2, 3)
  predicted: FAIL - Last dims 2 vs 3 incompatible.
  actual exception type: RuntimeError
  actual exception message: The size of tensor a (2) must match the size of tensor b (3) at non-singleton dimension 1

Invalid 5
  a (4, 1, 3) b (1, 2, 2)
  pr

**Why invalid examples fail:** broadcasting aligns dimensions from the right; aligned dimensions must be equal or one must be 1 (or missing). If both differ and neither is 1, expansion is undefined and PyTorch raises a runtime error.

## Summary
Sections 3-5 covered: indexing/slicing/selection, reshape/view/transpose/squeeze/unsqueeze/cat/stack, reshape validity, arithmetic/matmul/dot/matrix-vector/reductions/argmax, NumPy-PyTorch parity, and broadcasting rules with predicted valid/invalid experiments.
